# TA-03: Prediksi Risiko Gagal Bayar Kartu Kredit (Regresi Logistik)
**Nama:** Rian Surya | **NIM:** 301240058 | **Kelas:** 5C

Dataset: *Default of Credit Card Clients* (UCI / Kaggle). Letakkan file `UCI_Credit_Card.csv` di folder `data/`.

## 1. Import library dan load data

In [ ]:
import json, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print("Jalankan: pip install imbalanced-learn")

DATA_PATH = "data/UCI_Credit_Card.csv"
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"default.payment.next.month": "default",
                        "default payment next month": "default", "PAY_1": "PAY_0"})
print(df.shape)
df.head()

## 2. Pembersihan data
Kategori `EDUCATION` (0, 5, 6) dan `MARRIAGE` (0) tidak terdokumentasi pada deskripsi dataset, sehingga digabung ke kategori "lainnya".

In [ ]:
NUM_FEATURES = ["LIMIT_BAL", "AGE", "PAY_0", "PAY_2", "PAY_3", "BILL_AMT1", "PAY_AMT1"]
CAT_FEATURES = ["SEX", "EDUCATION", "MARRIAGE"]
CAT_LEVELS = [[1, 2], [1, 2, 3, 4], [1, 2, 3]]

df = df.drop(columns=[c for c in ["ID"] if c in df.columns]).drop_duplicates()
print("Missing value total:", int(df.isna().sum().sum()))
for c in NUM_FEATURES: df[c] = df[c].fillna(df[c].median())
for c in CAT_FEATURES: df[c] = df[c].fillna(df[c].mode()[0])
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})
df.describe().T

## 3. Exploratory Data Analysis

In [ ]:
counts = df["default"].value_counts().sort_index()
print(counts, f"\nRasio gagal bayar: {counts[1]/counts.sum():.2%}")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x="default", data=df, ax=axes[0]); axes[0].set_title("Distribusi target")
sns.boxplot(x="default", y="LIMIT_BAL", data=df, ax=axes[1]); axes[1].set_title("LIMIT_BAL vs default")
sns.barplot(x="PAY_0", y="default", data=df, ax=axes[2]); axes[2].set_title("Rasio gagal bayar per PAY_0")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[NUM_FEATURES + ["default"]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Korelasi fitur numerik"); plt.show()

## 4. Split data, scaling, dan penanganan class imbalance
Scaler di-*fit* hanya pada data latih, dan SMOTE hanya diterapkan pada data latih agar tidak terjadi *data leakage*.

In [ ]:
X = df[NUM_FEATURES + CAT_FEATURES]; y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(categories=CAT_LEVELS, handle_unknown="ignore"), CAT_FEATURES)])
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

if HAS_SMOTE:
    X_train_t, y_train_bal = SMOTE(random_state=42).fit_resample(X_train_t, y_train)
    class_weight = None
else:
    y_train_bal, class_weight = y_train, "balanced"
print("Distribusi data latih setelah penyeimbangan:", dict(pd.Series(y_train_bal).value_counts()))

## 5. Training model Regresi Logistik

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight=class_weight)
model.fit(X_train_t, y_train_bal)

## 6. Evaluasi model

In [ ]:
THRESHOLD = 0.5
proba = model.predict_proba(X_test_t)[:, 1]
pred = (proba >= THRESHOLD).astype(int)
print(classification_report(y_test, pred, target_names=["Tidak Gagal Bayar", "Gagal Bayar"]))

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
    display_labels=["Tidak Gagal Bayar", "Gagal Bayar"]).plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)
ax.set_title("Confusion Matrix"); plt.show()

In [ ]:
auc = roc_auc_score(y_test, proba)
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="orange", label=f"Kurva ROC (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="darkblue", label="Tebakan Acak")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Kurva ROC"); plt.legend(); plt.grid(True); plt.show()
print(f"AUC = {auc:.4f}")

### Dampak threshold terhadap Precision dan Recall
Dalam manajemen risiko, *False Negative* (nasabah berisiko diprediksi aman) umumnya lebih merugikan, sehingga Recall kelas "Gagal Bayar" perlu diperhatikan.

In [ ]:
rows = []
for th in (0.3, 0.4, 0.5, 0.6, 0.7):
    p = (proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    rows.append({"threshold": th, "precision": tp/max(tp+fp,1), "recall": tp/(tp+fn), "FP": fp, "FN": fn})
pd.DataFrame(rows).round(3)

In [ ]:
coef = pd.Series(model.coef_[0], index=preprocessor.get_feature_names_out()).sort_values()
coef.plot(kind="barh", figsize=(7, 6), title="Koefisien Regresi Logistik (fitur ter-scaling)"); plt.show()

## 7. Simpan model dan scaler dengan Joblib

In [ ]:
import os; os.makedirs("model", exist_ok=True)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
joblib.dump(model, "model/logreg_model.joblib")
joblib.dump(preprocessor, "model/scaler_preprocessor.joblib")
meta = {"num_features": NUM_FEATURES, "cat_features": CAT_FEATURES, "threshold": THRESHOLD,
        "metrics": {"auc": round(float(auc), 4), "recall": round(tp/(tp+fn), 4),
                    "precision": round(tp/max(tp+fp, 1), 4), "accuracy": round((tp+tn)/(tp+tn+fp+fn), 4)},
        "smote": HAS_SMOTE}
json.dump(meta, open("model/meta.json", "w"), indent=2)
print("Tersimpan:", os.listdir("model"))